In [3]:
from qiskit import QuantumCircuit

# --------------------------------------------------
# Prepare ancilla |A>
# |A> = (|0> + exp(i*pi/4)|1>)/sqrt(2)
# --------------------------------------------------
def gate_A(qc, A):
    qc.h(A)
    qc.t(A)


# --------------------------------------------------
# Temporary Logical AND Gate
# Inputs:
#   x, y, A
#
# A must already be prepared in |A>
# --------------------------------------------------
def gate_AND(qc, x, y, A):

    qc.cx(x, A)
    qc.cx(y, A)

    qc.cx(A, x)
    qc.cx(A, y)

    qc.barrier(x, y, A)

    qc.tdg(x)
    qc.tdg(y)
    qc.t(A)

    qc.cx(A, x)
    qc.cx(A, y)

    qc.h(A)
    qc.s(A)


# --------------------------------------------------
# Uncomputation Gate
# Inputs:
#   x, y, A
#
# A currently contains x·y
# --------------------------------------------------
def gate_AND_uncompute(qc, x, y, x_y):

    qc.sdg(x_y)
    qc.h(x_y)

    qc.cx(x_y, x)
    qc.cx(x_y, y)

    qc.barrier(x, y, x_y)

    qc.t(x)
    qc.t(y)
    

    qc.tdg(x_y)

    qc.cx(x_y, x)
    qc.cx(x_y, y)

    qc.cx(y,x_y)
    qc.cx(x,x_y)


In [4]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from numpy import pi

def build_seo_adder(A, B) :
    
    qreg_q = QuantumRegister(12, 'q')
    creg_c = ClassicalRegister(5, 'c')
    circuit = QuantumCircuit(qreg_q, creg_c)

    a_qubits = [9, 6, 3, 0]
    b_qubits = [10, 7, 4, 1]

    # ============================================================
    # ENCODE A
    # ============================================================
    
    for i in range(4):
        if (A >> i) & 1:
            circuit.x(qreg_q[a_qubits[3 - i]])
    
    # ============================================================
    # ENCODE B
    # ============================================================
    
    for i in range(4):
        if (B >> i) & 1:
            circuit.x(qreg_q[b_qubits[3 - i]])
    
    circuit.barrier();

    gate_A(circuit,qreg_q[2]);
    gate_A(circuit,qreg_q[5]);
    gate_A(circuit,qreg_q[8]);
    gate_A(circuit,qreg_q[11]);
    
    circuit.barrier();
    
    gate_AND(circuit, qreg_q[0], qreg_q[1], qreg_q[2]);
    
    circuit.barrier();
    
    circuit.cx(qreg_q[2], qreg_q[3]);
    circuit.cx(qreg_q[2], qreg_q[4]);
    
    circuit.barrier();
    
    gate_AND(circuit, qreg_q[3], qreg_q[4], qreg_q[5]);
    
    circuit.barrier();
    
    circuit.cx(qreg_q[2], qreg_q[5]);
    circuit.cx(qreg_q[5], qreg_q[6]);
    circuit.cx(qreg_q[5], qreg_q[7]);
    
    circuit.barrier();
    
    gate_AND(circuit, qreg_q[6], qreg_q[7], qreg_q[8]);
    
    circuit.barrier();
    
    circuit.cx(qreg_q[5], qreg_q[8]);
    circuit.cx(qreg_q[8], qreg_q[9]);
    circuit.cx(qreg_q[8], qreg_q[10]);
    
    circuit.barrier();
    
    gate_AND(circuit, qreg_q[9], qreg_q[10], qreg_q[11]);
    
    circuit.barrier();
    
    circuit.cx(qreg_q[8], qreg_q[11]);
    circuit.cx(qreg_q[8], qreg_q[9]);
    circuit.cx(qreg_q[5], qreg_q[8]);
    
    circuit.barrier();
    
    gate_AND_uncompute(circuit, qreg_q[6], qreg_q[7], qreg_q[8]);
    
    circuit.barrier();
    
    circuit.cx(qreg_q[5], qreg_q[6]);
    circuit.cx(qreg_q[2], qreg_q[5]);
    
    circuit.barrier();
    
    gate_AND_uncompute(circuit, qreg_q[3], qreg_q[4], qreg_q[5]);
    
    circuit.barrier();
    
    circuit.cx(qreg_q[2], qreg_q[3]);
    
    circuit.barrier();
    
    gate_AND_uncompute(circuit, qreg_q[0], qreg_q[1], qreg_q[2]);
    
    circuit.barrier();
    
    circuit.cx(qreg_q[0], qreg_q[1]);
    circuit.cx(qreg_q[3], qreg_q[4]);
    circuit.cx(qreg_q[6], qreg_q[7]);
    circuit.cx(qreg_q[9], qreg_q[10]);
    
    circuit.barrier();
    
    circuit.measure(qreg_q[1], creg_c[0]);
    circuit.measure(qreg_q[4], creg_c[1]);
    circuit.measure(qreg_q[7], creg_c[2]);
    circuit.measure(qreg_q[10], creg_c[3]);
    circuit.measure(qreg_q[11], creg_c[4]);
    
    return circuit

In [5]:
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit import transpile
from qiskit_aer import AerSimulator

p1 = 0.01  # 1-qubit gate error
p2 = 0.02   # 2-qubit gate error

error_1q = pauli_error([
    ('X', p1),
    ('I', 1 - p1)
])

error_2q_single = pauli_error([
    ('X', p2),
    ('I', 1 - p2)
])

error_2q = error_2q_single.tensor(error_2q_single)

noise_model = NoiseModel()

noise_model.add_all_qubit_quantum_error(
    error_1q,
    ["x", "h", "s", "sdg", "t", "tdg"]
)

noise_model.add_all_qubit_quantum_error(
    error_2q,
    ['cx']
)

In [6]:
shots = 1000

# ====================================================
# CREATE NOISY SIMULATOR
# ====================================================

sim = AerSimulator(noise_model=noise_model)
results = []
for a in range(16):
    for b in range(16):
        qc = build_seo_adder(a, b)
        compiled = transpile(qc, sim)
        counts = sim.run(compiled, shots=shots).result().get_counts()

        correct = a + b
        correct_output = format(correct, '05b')   # 5-bit binary

        correct_counts = counts.get(correct_output, 0)
        ER = 1 - correct_counts / shots

        total_ED = 0
        total_relative_ED = 0
        for output, freq in counts.items():
            noisy_decimal = int(output, 2)
            ED = abs(correct - noisy_decimal)
            total_ED += ED * freq
            if correct != 0:
                total_relative_ED += (ED / correct) * freq

        mean_ED = total_ED / shots
        D = 30
        NMED = mean_ED / D
        MRED = total_relative_ED / shots

        results.append((a, b, correct_output, counts, ER, NMED, MRED))

In [7]:
sum_er = sum(item[4] for item in results)
sum_nmed = sum(item[5] for item in results)
sum_mred = sum(item[6] for item in results)
er = sum_er/256
nmed = sum_nmed/256
mred = sum_mred/256

print("Bitflip Noise\n")
print(f"\nER(%) : {er*100}")
print(f"\nNMED : {nmed}")
print(f"\nMRED : {mred}")

Bitflip Noise


ER(%) : 81.5265625

NMED : 0.16775130208333333

MRED : 0.5022422145303217
